## `optimade-maker` tutorial

This tutorial guides users through the main functionality of the toolkit. The goal is to start an OPTIMADE API server from raw crystal structure data, and demonstrate how to use the API.

### 1. Source data

The tutorial source data consists of 

- `./cifs/` - folder containing CIFs (Crystallographic Information Files) downloaded from the MC3D (Materials Cloud 3D Structures Database);
- `./properties.csv` -- file containing the densities for each structure (in kg/m^3).

### 2. Installing the toolkit

In order to start the OPTIMADE API from this data, one should install the toolkit with:

```bash
> pip install optimade-maker[tutorial]
```

(note that the `[tutorial]` extra dependency just installs support for this jupyter notebook).

### 3. Preparing the `optimade.yaml`

To make the source data understandable by `optimade-maker`, one needs to prepare an `optimade.yaml` configuration file. The Pydantic schema of this configuration file is available in the `config.py` file of the library. For this tutorial, the file has already been prepared, but we'll explain it by blocks below.

```yaml
config_version: 0.2.0
database_description: Simple DB
```

The first two lines contain the version of the config file, and a string to describe the dataset. The config version, in most cases, should typically match the latest version defined by the schema (either checked from `config.py` or the main README).

```yaml
entries:
  - entry_type: structures
    entry_paths:
      - path: cifs
        matches: ["*.cif"]
```

The `entries:` block contains the description of OPTIMADE entries (e.g. structures, references, ...). In this tutorial, we are serving only crystal structures, which is described by the `entry_type: structures` line, and the `entry_paths:` describe where to look for the structures. Here, we only define a single path, the `cifs` directory, and look for files that match the `*.cif` pattern.

```yaml
    property_paths:
      - path: properties.csv
    property_definitions:
      - name: density
        title: Density
        description: Density of the material
        unit: kg/m^3
        type: float
```

This block (nested inside `entry_type: structures`) shows which file contains the custom properties, and contains its metadata.

### 4. Starting the OPTIMADE API

We are now ready to start the OPTIMADE API from this dataset. To do this, run the `optimake` CLI in a separate terminal in the tutorial folder:

```bash
> optimake serve .
2026-04-20 17:03:03 INFO optimade-maker: optimade.jsonl doesn't exist. Converting archive.
Parsing structures files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 206.09it/s]
Constructing OPTIMADE structures entries: 3it [00:00, 1847.44it/s]
Parsing property files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 365.33it/s]
2026-04-20 17:03:04 INFO optimade-maker: Preparing to start the API...
2026-04-20 17:03:04 INFO optimade-maker: Using the MongoMock backend.
INFO:     [optimade] Using: Mock MongoDB (mongomock) @ localhost:27017
2026-04-20 17:03:04 INFO optimade-maker: Populating the database...
2026-04-20 17:03:04 INFO optimade-maker: Inserted 3 rows from the JSONL file
2026-04-20 17:03:04 INFO optimade-maker: Starting the API
INFO:     [optimade] Loaded settings from /home/kristjan/.optimade.json
INFO:     Started server process [1752662]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
```

Note, for other functionality and options of the CLI, see `optimake --help`.

### 5. Querying the API through Python

While the API is running in the background, you can query it according to the OPTIMADE specification (https://www.optimade.org/specification). The following cells show a minimal example on how to do this with Python (converting the structures to ASE (Atomic Simulation Environment) format):

In [1]:
import requests
from optimade.adapters import Structure

BASE_URL = "http://127.0.0.1:5000"


def get_structures(filter_):
    response = requests.get(f"{BASE_URL}/structures?filter={filter_}")
    response.raise_for_status()
    return [Structure(entry).as_ase for entry in response.json()["data"]]

In [2]:
# OPTIMADE query: Get all structures with more than 1 element
for structure in get_structures("nelements > 1"):
    print(structure)

Atoms(symbols='BaTiO3', pbc=True, cell=[3.9745754582274, 3.9745754582274, 3.9745754582274])
Atoms(symbols='Sr2C12', pbc=True, cell=[[4.3196827147571, 0.0, 0.0], [-2.1598413573785002, 3.740954967258099, 0.0], [0.0, 0.0, 9.7058202670626]])


In [3]:
# OPTIMADE query: filter based on the custom density property
# Note that the default prefix for custom properties is "_optimake_" for optimade-maker
for structure in get_structures("_optimake_density > 3800"):
    print(structure)

Atoms(symbols='Ba', pbc=True, cell=[[4.22762646696287, 0.0, 0.0], [-1.4092088223209558, 3.9858444574842284, 0.0], [-1.4092088223209558, -1.9929222287421124, 3.4518425557147467]])
Atoms(symbols='BaTiO3', pbc=True, cell=[3.9745754582274, 3.9745754582274, 3.9745754582274])


### 6. Using the API with an external client

The local API can also be queried with existing clients, such as the Materials Cloud OPTIMADE Client (https://optimadeclient.materialscloud.io), which allows to connect to a custom API URL.

To try it out, while the API is running in the background, open: https://optimadeclient.materialscloud.io/?base_url=http://127.0.0.1:5000